# Phase 2: Ragas Evaluation

> Requires Phase 1 to have been run in the same session (or cells re-run below).


## 1. Imports and Setup


In [ ]:
#@title Install dependencies
!pip install -q ragas datasets langchain langchain-community langchain-openai tiktoken
print('✓ Dependencies installed')


In [ ]:
#@title Configuration { display-mode: "form" }
import os
import pandas as pd
from datasets import Dataset

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

OPENAI_API_KEY = ""  #@param {type:"string"}
OPENAI_MODEL = "gpt-4o-mini"  #@param {type:"string"}

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Ragas uses its own LLM/embeddings — point to lightweight models
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_llm = LangchainLLMWrapper(ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", OPENAI_MODEL),
    temperature=0
))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

faithfulness.llm = ragas_llm
answer_relevancy.llm = ragas_llm
answer_relevancy.embeddings = ragas_embeddings

print('✓ Ragas configured with model:', os.getenv('OPENAI_MODEL', OPENAI_MODEL))


## 2. Validate Phase 1 Objects Exist


In [ ]:
required_columns = {"question", "retrieved_context", "generated_answer"}

assert "baseline_df" in globals(), "baseline_df from Phase 1 is required"
assert "degraded_df" in globals(), "degraded_df from Phase 1 is required"

assert required_columns.issubset(baseline_df.columns), f"baseline_df missing: {required_columns - set(baseline_df.columns)}"
assert required_columns.issubset(degraded_df.columns), f"degraded_df missing: {required_columns - set(degraded_df.columns)}"

print('✓ Phase 1 objects validated')
print('  baseline_df rows:', len(baseline_df))
print('  degraded_df rows:', len(degraded_df))


## 3. Convert RAG Outputs to Ragas Format


In [ ]:
def to_ragas_dataset(df: pd.DataFrame) -> Dataset:
    ragas_df = pd.DataFrame({
        "question": df["question"],
        "answer": df["generated_answer"],
        "contexts": df["retrieved_context"].apply(lambda x: [x] if isinstance(x, str) else x)
    })
    return Dataset.from_pandas(ragas_df)

print('✓ Conversion helper defined')


## 4. Create Ragas Datasets


In [ ]:
baseline_ragas_dataset = to_ragas_dataset(baseline_df)
degraded_ragas_dataset = to_ragas_dataset(degraded_df)

print('✓ baseline_ragas_dataset:', baseline_ragas_dataset)
print('✓ degraded_ragas_dataset:', degraded_ragas_dataset)


## 5. Evaluate Baseline Outputs


In [ ]:
baseline_eval = evaluate(
    baseline_ragas_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
    ],
)

baseline_scores_df = baseline_eval.to_pandas()
baseline_scores_df


## 6. Evaluate Degraded Outputs


In [ ]:
degraded_eval = evaluate(
    degraded_ragas_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
    ],
)

degraded_scores_df = degraded_eval.to_pandas()
degraded_scores_df


## 7. Compare Baseline vs Degraded Scores


In [ ]:
score_comparison_df = pd.DataFrame({
    "question": baseline_df["question"].values,
    "baseline_answer": baseline_df["generated_answer"].values,
    "degraded_answer": degraded_df["generated_answer"].values,
    "baseline_faithfulness": baseline_scores_df["faithfulness"].values,
    "degraded_faithfulness": degraded_scores_df["faithfulness"].values,
    "baseline_answer_relevancy": baseline_scores_df["answer_relevancy"].values,
    "degraded_answer_relevancy": degraded_scores_df["answer_relevancy"].values,
})

score_comparison_df["faithfulness_delta"] = (
    score_comparison_df["degraded_faithfulness"]
    - score_comparison_df["baseline_faithfulness"]
)

score_comparison_df["answer_relevancy_delta"] = (
    score_comparison_df["degraded_answer_relevancy"]
    - score_comparison_df["baseline_answer_relevancy"]
)

score_comparison_df


## 8. Aggregate Scores


In [ ]:
aggregate_scores_df = pd.DataFrame({
    "run": ["baseline", "degraded"],
    "avg_faithfulness": [
        baseline_scores_df["faithfulness"].mean(),
        degraded_scores_df["faithfulness"].mean(),
    ],
    "avg_answer_relevancy": [
        baseline_scores_df["answer_relevancy"].mean(),
        degraded_scores_df["answer_relevancy"].mean(),
    ],
})

aggregate_scores_df


## 9. Faithfulness Failure Case


In [ ]:
# Answer sounds correct but asserts a specific year/org not in the context — model hallucinated.
faithfulness_failure_dataset = Dataset.from_dict({
    "question": [
        "What is retrieval-augmented generation?"
    ],
    "answer": [
        "Retrieval-augmented generation was originally invented in 2020 by Meta AI and combines retrieval with text generation."
    ],
    "contexts": [
        [
            "Retrieval-augmented generation is a technique that improves language model responses by retrieving relevant external information before generation."
        ]
    ],
})

faithfulness_failure_eval = evaluate(
    faithfulness_failure_dataset,
    metrics=[faithfulness],
)

faithfulness_failure_eval.to_pandas()


## 10. Answer Relevance Failure Case


In [ ]:
# Context contains the correct answer but the model responds with unrelated AI generalities.
answer_relevance_failure_dataset = Dataset.from_dict({
    "question": [
        "How does RAG help reduce hallucinations?"
    ],
    "answer": [
        "Artificial intelligence is a broad field with many applications. It includes machine learning, robotics, expert systems, and natural language processing. Many companies use AI for automation and analytics."
    ],
    "contexts": [
        [
            "RAG helps reduce hallucinations by grounding generated answers in retrieved external documents instead of relying only on the model's internal parameters."
        ]
    ],
})

answer_relevance_failure_eval = evaluate(
    answer_relevance_failure_dataset,
    metrics=[answer_relevancy],
)

answer_relevance_failure_eval.to_pandas()


## 11. Export Results


In [ ]:
baseline_scores_df.to_csv("phase2_baseline_ragas_scores.csv", index=False)
degraded_scores_df.to_csv("phase2_degraded_ragas_scores.csv", index=False)
score_comparison_df.to_csv("phase2_score_comparison.csv", index=False)
aggregate_scores_df.to_csv("phase2_aggregate_scores.csv", index=False)

print('✓ Exported:')
print('  phase2_baseline_ragas_scores.csv')
print('  phase2_degraded_ragas_scores.csv')
print('  phase2_score_comparison.csv')
print('  phase2_aggregate_scores.csv')


## 12. Final Summary


In [ ]:
print("Aggregate Ragas Scores")
display(aggregate_scores_df)

print("Baseline vs Degraded Score Comparison")
display(score_comparison_df)
